# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access and print top-level metadata
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
# Optionally, print available metadata attributes
print("Available metadata attributes:")
print([a for a in dir(dataset.metadata) if not a.startswith('_')])

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's list all record sets in the dataset and inspect their fields/columns using the Croissant `@id` values. We'll use the dataset's schema to discover the available structure programmatically.

In [ ]:
# List all record sets and their field/column IDs
record_sets = dataset.metadata.record_sets
print(f"Found {len(record_sets)} record set(s).\n")

for rset in record_sets:
    print(f"Record Set Name: {getattr(rset, 'name', None)} | @id: {rset.id}")
    # List columns (fields) for each record set
    fields = getattr(rset, 'fields', [])
    if not fields:
        # Some datasets use .columns instead of .fields
        fields = getattr(rset, 'columns', [])
    for fld in fields:
        print(f"    Field/Column: {getattr(fld, 'name', None)} | @id: {fld.id} | Type: {getattr(fld, 'data_type', None)}")
    print()

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis.

Use the `@id` of the record set and columns/fields you found above to extract the data. The code below pulls all record sets into pandas DataFrames and prints the column names and the first few rows for inspection.

In [ ]:
# Build a list of record set @ids to extract
record_set_ids = [rset.id for rset in dataset.metadata.record_sets]

# Create a dictionary of DataFrames
dataframes = {}
for rset_id in record_set_ids:
    # Load records into a DataFrame
    df = pd.DataFrame(list(dataset.records(record_set=rset_id)))
    dataframes[rset_id] = df

# Print all loaded DataFrames and their columns
for rset_id, df in dataframes.items():
    print(f"Record Set @id: {rset_id}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(), '\n')

## 4. Exploratory Data Analysis (EDA)
Let's perform EDA using one of the main record sets (e.g., the survey responses).

**Example:** We'll filter based on participant age (or the `GAD-7` total score, if available), normalize a numeric field, and group by a categorical field (such as gender or village).

*Update the variables below (`target_rset_id`, `numeric_field_id`, `group_field_id`) to use the relevant `@id`s from the overview. If unsure, look at the printed record set and field IDs above.*

In [ ]:
# Configuration: Update these based on your schema exploration
target_rset_id = record_set_ids[0]  # Assume the first record set is the main survey table

df = dataframes[target_rset_id]
# Try to auto-detect a numeric field (e.g., age, GAD-7, PHQ-9, or PSQ score)
numeric_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] or 'score' in col.lower() or 'age' in col.lower()]  # heuristic
print(f"Numeric candidates: {numeric_candidates}")

if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Using numeric field: {numeric_field}")

    # Example filter: Show records with numeric_field > threshold
    threshold = df[numeric_field].mean() if df[numeric_field].dtype != 'O' else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field (e.g., gender)
    group_candidates = [col for col in df.columns if df[col].dtype == 'O' and col != numeric_field]
    group_field = group_candidates[0] if group_candidates else None
    if group_field:
        print(f"\nGrouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(grouped_df.head())
else:
    print("No obvious numeric field found. Please check the columns to proceed with EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we show an example histogram for the main numeric field and a boxplot grouped by a categorical field. Update the variables as appropriate for your context.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'numeric_field' in locals() and numeric_field in df.columns:
    plt.figure(figsize=(7,4))
    df[numeric_field].hist(bins=20)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group_field if available
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.xticks(rotation=30)
        plt.title(f'{numeric_field} by {group_field}')
        plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion
We have explored the FAIR² Mental Health Survey Dataset using Croissant metadata and the `mlcroissant` library. 

Key findings include the ability to:
- Inspect schema details using @id references for all entities
- Flexibly extract and explore any record set
- Apply standard data processing and visualization tasks on the loaded DataFrames.

This workflow can be adapted for any Croissant-compatible dataset. For further analysis, consider integrating advanced statistics or machine learning pipelines.